# Lab 6.1 — Write an Eval Suite

*Chapter 6 — Evaluating GenAI Systems · 45 minutes · JupyterLab + the OpenAI API*

GenAI has no compiler for correctness — evaluation is the harness. You will
build the smallest useful one: a **golden dataset** for a code-review-comment
generator, **two scorers** (a deterministic programmatic check and an
LLM-as-judge rubric), and a **regression gate** that catches a prompt change
making the system worse. This is Chapter 5's TDD moved up one level: the eval
suite is to a prompt what pytest is to a function.

Everything runs offline: with `COURSE_AI_MOCK=1` (or no key) the mock in
`course_ai` deterministically produces a good baseline *and* the regressed
broken output, so the suite demonstrably catches the regression with no key.

## Objectives

By the end of this lab, you will:

- Assemble a golden dataset: realistic inputs, expectations, adversarial cases.
- Implement two scorers — a programmatic assertion and a 1–5 judge rubric —
  and compare their verdicts.
- Run a baseline prompt and a broken variant through the suite and catch the
  regression with a gate you could drop into CI.

## Setup

- **Key:** `OPENAI_API_KEY` from the environment / course `.env`, never
  printed. `COURSE_AI_MOCK=1` (or no key) engages the deterministic mock.
- **Model:** pinned via `OPENAI_MODEL` (default `gpt-4o-mini`); the judge uses
  the same pin — in production you would consider a separate judge model.
- **Artifact under test:** a prompt that turns a code diff into structured
  review comments (`{comments: [{line, severity, message}]}`).

In [ ]:
import json
import pathlib

import course_ai
from course_ai import chat, chat_json

print("mode:", course_ai.mode())

## Steps

### Step 1 — The golden dataset (10 min)

14 cases, provided: twelve diffs with a planted defect each, two clean diffs.
Each case carries **expectations** — the keywords a correct review must
mention (any one is enough), or `clean: true` for diffs that must come back
with no high/medium findings. Every bug you have ever seen in review or
production belongs in a set like this; the dataset only grows.

In [ ]:
GOLDEN = [
    {"id": "C01", "expect": ["except", "specific"], "diff":
        "def load_config(path):\n    try:\n        return json.load(open(path))\n    except:\n        pass"},
    {"id": "C02", "expect": ["true", "boolean", "truth"], "diff":
        "def is_ready(flags):\n    if flags[\"ready\"] == True:\n        return True\n    return False"},
    {"id": "C03", "expect": ["password", "secret", "credential"], "diff":
        "DB_URL = \"postgres://db.internal/app\"\npassword = \"hunter2\"  # temporary until vault wired up"},
    {"id": "C04", "expect": ["eval", "arbitrary"], "diff":
        "def apply_rule(expr, ctx):\n    return eval(expr, {}, ctx)"},
    {"id": "C05", "expect": ["mutable", "default"], "diff":
        "def add_tag(tag, tags=[]):\n    tags.append(tag)\n    return tags"},
    {"id": "C06", "expect": ["todo"], "diff":
        "def retry(request):\n    # TODO: exponential backoff before the launch\n    return send(request)"},
    {"id": "C07", "expect": ["print", "logging"], "diff":
        "def discount(price, pct):\n    print(\"computing discount\")\n    return price * (1 - pct / 100)"},
    {"id": "C08", "expect": ["sql", "parameter", "injection"], "diff":
        "def find_user(uid):\n    q = f\"SELECT * FROM users WHERE id = {uid}\"\n    return db.execute(q)"},
    {"id": "C09", "expect": ["assert", "raise", "-o"], "diff":
        "def set_quota(user, gb):\n    assert gb > 0\n    user.quota = gb"},
    {"id": "C10", "expect": ["wildcard", "import", "explicit"], "diff":
        "from utils import *\n\ndef render(page):\n    return html(page)"},
    {"id": "C11", "expect": ["pickle", "deserialize", "untrusted"], "diff":
        "def restore_session(blob):\n    return pickle.loads(blob)"},
    {"id": "C12", "expect": ["pass", "log", "raise", "silently"], "diff":
        "def refresh_cache():\n    try:\n        cache.rebuild()\n    except Exception as e:\n        pass"},
    {"id": "C13", "expect": [], "clean": True, "diff":
        "def area(w, h):\n    if w <= 0 or h <= 0:\n        raise ValueError(\"sides must be positive\")\n    return w * h"},
    {"id": "C14", "expect": [], "clean": True, "diff":
        "def chunk(items, size):\n    for i in range(0, len(items), size):\n        yield items[i:i + size]"},
]
for c in GOLDEN:
    c.setdefault("clean", False)
print(f"{len(GOLDEN)} golden cases: {sum(c['clean'] for c in GOLDEN)} clean, "
      f"{sum(not c['clean'] for c in GOLDEN)} with planted defects")

# YOUR CODE (optional): add one case from a bug you have personally seen in
# review. Give it the keywords a correct review would have to mention.

### Step 2 — The system under test: baseline vs broken prompt (5 min)

Two prompt variants drive the same generator. `BASELINE` carries the format
contract and quality bar; `BROKEN` is the plausible "small cleanup" someone
ships on a Friday — it drops both. The suite's job is to prove that edit is
not small.

In [ ]:
BASELINE_PROMPT = (
    "You are a meticulous staff engineer doing a code review. "
    "Respond with JSON only: {\"comments\": [{\"line\": int, \"severity\": "
    "\"high|medium|low|info\", \"message\": \"specific and actionable\"}]}. "
    "Name the exact problem and the fix. If the diff is clean, return one "
    "info-level comment saying so."
)
BROKEN_PROMPT = "Take a quick look and say what you think."

def generate_review(diff, prompt_template):
    """The system under test: one chat call per case."""
    return chat(prompt_template
                + "\n\nReview the following diff:\n```diff\n" + diff + "\n```")

sample = generate_review(GOLDEN[0]["diff"], BASELINE_PROMPT)
print(sample)

### Step 3 — Scorer 1: the programmatic check (10 min)

Cheap, deterministic, runs on every case: does the output parse as JSON, does
it match the schema, and does it mention what a correct review must mention
(or stay quiet on a clean diff)? Complete `score_programmatic` — the skeleton
and a reference behavior are described in the docstring.

In [ ]:
def extract_json(raw):
    """Pull the JSON object out of a model reply (handles prose, fences,
    and the [MOCK] prefix). Returns None when there is no object at all."""
    start, end = raw.find("{"), raw.rfind("}")
    if start == -1 or end <= start:
        return None
    try:
        return json.loads(raw[start:end + 1])
    except json.JSONDecodeError:
        return None

def score_programmatic(case, raw):
    """Return (passed: bool, reasons: list[str]).

    Checks:
      1. output contains a parseable JSON object
      2. it has a 'comments' list of {line: int, severity: str, message: str}
      3. defect cases: some expected keyword appears in a message (any case)
      4. clean cases: no comment with severity high or medium
    """
    reasons = []
    obj = extract_json(raw)
    if not isinstance(obj, dict):
        return False, ["no parseable JSON object in the output"]
    comments = obj.get("comments")
    if not isinstance(comments, list):
        return False, ["missing 'comments' list"]
    for c in comments:
        if not (isinstance(c, dict) and isinstance(c.get("line"), int)
                and isinstance(c.get("severity"), str) and isinstance(c.get("message"), str)):
            reasons.append("a comment is missing line/severity/message")
            break
    text = " ".join(str(c.get("message", "")) for c in comments).lower()
    if case["clean"]:
        loud = [c for c in comments if str(c.get("severity", "")).lower() in {"high", "medium"}]
        if loud:
            reasons.append(f"clean diff flagged {len(loud)} high/medium finding(s)")
    else:
        if not any(k.lower() in text for k in case["expect"]):
            reasons.append(f"none of the expected keywords {case['expect']} found in messages")
    return not reasons, reasons

# smoke-test the scorer before trusting it: a good output and a bad one
ok_good = score_programmatic(GOLDEN[0], '{"comments": [{"line": 4, "severity": "high", "message": "bare except"}]}')
ok_bad = score_programmatic(GOLDEN[0], "Looks fine to me.")
print("scorer smoke test — good output passes:", ok_good[0], "| bad output fails:", not ok_bad[0])
assert ok_good[0] and not ok_bad[0], "the scorer itself is broken — fix it before trusting the suite"

### Step 4 — Scorer 2: LLM-as-judge (5 min)

The programmatic check verifies shape and keywords; it cannot tell a specific
finding from a lucky guess. The judge reads the diff and the candidate review
and scores 1–5 on a rubric. Judges are models too — version this prompt, and
calibrate it against human review before you trust it alone.

In [ ]:
JUDGE_RUBRIC = (
    "Rate the review 1-5: 5 = every real defect found with a specific, actionable "
    "message and no invented issues; 3 = findings present but vague or partially "
    "wrong; 1 = vague approval, wrong claims, or unstructured prose. "
    "Answer as JSON with keys score (integer 1-5) and rationale (one sentence)."
)
JUDGE_SCHEMA = {"type": "object",
                "properties": {"score": {"type": "integer"}, "rationale": {"type": "string"}},
                "required": ["score", "rationale"]}

def score_judge(case, raw):
    prompt = (JUDGE_RUBRIC + "\n\nCase diff:\n```diff\n" + case["diff"]
              + "\n```\n\nREVIEW UNDER TEST:\n" + raw)
    out = chat_json(prompt, schema=JUDGE_SCHEMA)
    return int(out["score"]), out["rationale"]

score, why = score_judge(GOLDEN[0], sample)
print(f"judge on the sample baseline output: {score}/5 — {why}")

### Step 5 — Run the suite: baseline vs broken (10 min)

One function runs a prompt variant over every golden case with both scorers.
Then the scorecard — and the gate. Watch the broken variant get caught by
*both* scorers: the programmatic one sees the format collapse, the judge sees
the quality collapse.

In [ ]:
def run_suite(prompt_template, cases=GOLDEN):
    rows = []
    for case in cases:
        raw = generate_review(case["diff"], prompt_template)
        ok, reasons = score_programmatic(case, raw)
        judge, why = score_judge(case, raw)
        rows.append({"id": case["id"], "ok": ok, "reasons": reasons,
                     "judge": judge, "why": why})
    return rows

def summarize(label, rows):
    prog = sum(r["ok"] for r in rows) / len(rows)
    judge = sum(r["judge"] for r in rows) / len(rows)
    print(f"\n=== {label} ===")
    for r in rows:
        flag = "PASS" if r["ok"] else "FAIL"
        print(f"  {r['id']}: programmatic {flag:4} | judge {r['judge']}/5"
              + ("" if r["ok"] else f"  <- {'; '.join(r['reasons'])}"))
    print(f"  --> programmatic pass rate {prog:.0%} | judge mean {judge:.1f}/5")
    return {"label": label, "programmatic": prog, "judge_mean": judge, "rows": rows}

baseline = summarize("BASELINE prompt", run_suite(BASELINE_PROMPT))
broken = summarize("BROKEN prompt (the Friday cleanup)", run_suite(BROKEN_PROMPT))

### Step 6 — The regression gate and the scorecard (5 min)

This is the exit criterion: the suite must catch the regression, loudly.
Thresholds are loose on purpose — a live model is stochastic; the *direction*
and *size* of the gap is the signal, not any single case.

In [ ]:
drop_prog = baseline["programmatic"] - broken["programmatic"]
drop_judge = baseline["judge_mean"] - broken["judge_mean"]
print(f"programmatic pass-rate drop: {drop_prog:.0%}")
print(f"judge mean drop:             {drop_judge:.1f} points")

assert drop_prog >= 0.4, "the suite did NOT catch the format regression — check the scorers"
assert drop_judge >= 1.0, "the judge did NOT catch the quality regression — calibrate it"
print("\nREGRESSION CAUGHT — this gate is ready to run in CI on every prompt change.")

scorecard = {"cases": len(GOLDEN),
             "baseline": {k: v for k, v in baseline.items() if k != "rows"},
             "broken": {k: v for k, v in broken.items() if k != "rows"},
             "gate": {"programmatic_drop": drop_prog, "judge_drop": drop_judge}}
pathlib.Path("eval_scorecard.json").write_text(json.dumps(scorecard, indent=2))
print("scorecard written to eval_scorecard.json")

### Step 7 — Compare the scorers (5 min)

The two scorers measure different things. Read the per-case rows and answer:
where could the programmatic check pass but the judge object? (Valid JSON,
right keywords, wrong diagnosis.) Where could the judge be generous but the
check fail? (Great prose, no schema.) That is why evals layer scorers instead
of picking one.

In [ ]:
print(f"{'case':6} {'programmatic':14} {'judge':6} verdict")
for b, r in zip(baseline["rows"], broken["rows"]):
    print(f"{b['id']:6} {str(b['ok']):14} {b['judge']}/5   "
          f"{'both scorers agree' if b['ok'] == (r['judge'] >= 3) else 'inspect this case'}")

# YOUR CODE (stretch): fix BROKEN_PROMPT — restore the format contract and the
# quality bar — and re-run run_suite on your fixed prompt until the gate passes.

## Deliverable

1. The scorecard (`eval_scorecard.json`) showing baseline vs broken on both
   scorers.
2. The passing regression gate (Step 6's assertions).
3. One sentence: which scorer caught the regression first, and what each
   scorer is blind to.

## Reflection

1. Fourteen cases caught this regression. What regression would they NOT
   catch? (Think: tone, language, a defect type not in the set.)
2. What does it cost to run this suite on every prompt edit — and what does
   the alternative cost?
3. The judge is a model. How would you calibrate it before trusting it alone
   in CI?

## Debrief (instructor-led)

1. Compare scorecards: did anyone's live run fail a case the mock passed?
   Which case — and is the case or the model wrong?
2. The golden set grows from real bugs. What is the intake process on your
   team for "production incident → golden case"?
3. Where does this suite plug into your pipeline: pre-merge gate, nightly,
   or release checklist? Who owns a red bar?
4. Bridge to Chapter 7: agents fail in their *trajectories*, not just their
   answers. What would a golden case for an agent look like?

## Troubleshooting

- **The mock baseline scores 100% and the mock broken variant scores 0%** —
  expected: the mock is deterministic and produces both extremes by design,
  so the regression is visible with no key. Live runs are noisier; the gate
  thresholds tolerate that.
- **`extract_json` returns None on a live reply** — the model answered in
  prose. Strengthen the format contract ("JSON only, no prose") and consider
  `response_format` / the Responses API `text.format` option from Lab 4.1.
- **A clean case fails with "high/medium finding"** — either the model
  invented an issue (a real finding about your prompt) or the case is not as
  clean as you thought. Inspect before you edit the case.
- **Judge scores look flat (all 3s) on a live run** — the rubric is too vague.
  Sharpen the anchors (what exactly earns a 5?) and add two calibrated
  examples.
- **The gate assertion fires on the baseline itself** — the suite, not the
  prompt, may be broken. Re-run the Step 3 scorer smoke test first; never
  debug a prompt against a scorer you have not validated.